In [24]:
import pandas as pd 
import numpy as np

In [25]:
df=pd.read_csv("C:\\Users\\Jathin Prakash\\Desktop\\final_dataset.csv")

In [26]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
movies = df[["title", "tags"]]

movies.head()

,title,tags
0,Toy Story,Animation Comedy Family jealousy toy boy frien...
1,Jumanji,Adventure Fantasy Family boardgame disappearan...
2,Grumpier Old Men,Romance Comedy fishing bestfriend duringcredit...
3,Father of the Bride Part II,Comedy baby midlifecrisis confidence aging dau...
4,Heat,Action Crime Drama Thriller robbery detective ...


In [28]:
import re

def clean(text):
    text=str(text)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

movies['tags'] = movies['tags'].apply(clean)

C:\Users\Jathin Prakash\AppData\Local\Temp\ipykernel_19972\4024048758.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies['tags'] = movies['tags'].apply(clean)


In [29]:
movies=movies.dropna()

In [30]:
len(movies)

9185

In [14]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to C:\Users\Jathin
[nltk_data]     Prakash\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Jathin
[nltk_data]     Prakash\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [31]:
from nltk.stem.porter import PorterStemmer
import nltk

# Download once
nltk.download('punkt')

ps = PorterStemmer()

# Apply stemming
movies['tags'] = movies['tags'].apply(
    lambda x: " ".join([ps.stem(word) for word in x.split()])
)

[nltk_data] Downloading package punkt to C:\Users\Jathin
[nltk_data]     Prakash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [32]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1,2)
)

vectors = tfidf.fit_transform(movies['tags'])

print(vectors.shape)

(9185, 15000)


In [33]:
similarity = cosine_similarity(vectors)

print(similarity.shape)

(9185, 9185)


In [34]:
def recommend(movie_name, top_n=5):
    
    # Check whether movie exists
    if movie_name not in movies['title'].values:
        print("Movie not found in dataset")
        return

    # Get movie index
    movie_index = movies[movies['title'] == movie_name].index[0]

    # Get similarity scores
    distances = similarity[movie_index]

    # Sort movies based on similarity
    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:top_n+1]
    print(f"\nRecommended movies for '{movie_name}':\n")

    for i, movie in enumerate(movie_list, start=1):
        index = movie[0]
        score = movie[1]

        print(f"{i}. {movies.iloc[index].title}  | Similarity Score: {score:.4f}")

In [35]:
recommend("Interstellar")


Recommended movies for 'Interstellar':

1. Apollo 13  | Similarity Score: 0.2509
2. Love  | Similarity Score: 0.2284
3. The Martian  | Similarity Score: 0.2190
4. Silent Running  | Similarity Score: 0.1899
5. Two Is a Family  | Similarity Score: 0.1765


In [20]:
import pickle

pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [36]:
recommend("Titanic")


Recommended movies for 'Titanic':

1. Love Story  | Similarity Score: 0.1929
2. Romeo + Juliet  | Similarity Score: 0.1620
3. Sidewalls  | Similarity Score: 0.1527
4. A Walk to Remember  | Similarity Score: 0.1489
5. Threads  | Similarity Score: 0.1418


In [33]:
movies

,title,tags
0,Toy Story,anim comedi famili jealousi toy boy friendship...
1,Jumanji,adventur fantasi famili boardgam disappear bas...
2,Grumpier Old Men,romanc comedi fish bestfriend duringcreditssti...
3,Father of the Bride Part II,comedi babi midlifecrisi confid age daughter m...
4,Heat,action crime drama thriller robberi detect ban...
...,...,...
9180,With Open Arms,comedi patriceledoux christianclavi aryabittan...
9181,The Visitors: Bastille Day,comedi nazi castl timetravel robespierr jean-m...
9182,Titanic 2,action adventur thriller suspens shanevandyk s...
9183,In a Heartbeat,famili anim romanc comedi love teenag lgbt sho...


In [37]:
recommend("The Visitors: Bastille Day")


Recommended movies for 'The Visitors: Bastille Day':

1. Just Visiting  | Similarity Score: 0.3908
2. Junior  | Similarity Score: 0.2288
3. The Round Up  | Similarity Score: 0.2256
4. The Corsican File  | Similarity Score: 0.2249
5. The Visitors II: The Corridors of Time  | Similarity Score: 0.2173
